# 횟수 검증 재감사: 개체별 상태 전환

이전 SmartHome-Bench 실험 6개와 `dog_escape`를 같은 기준으로 다시 정리했습니다. 저장된 모델 응답과 프레임 검토 기록을 읽는 오프라인 노트북이며 API/DB 호출이 없습니다.

**계수 기준:** 각 동물에 대해 `이전 상태 → 목표 상태`가 영상에서 확인되면 1회입니다. 같은 상태가 이어지는 동안의 모델 중복 행은 1회로 묶습니다. 같은 동물의 다음 회수는 목표 상태를 떠나 이전 상태로 돌아간 장면이 확인되어야 합니다. 영상 시작부터 이미 목표 상태이거나, 전환 전후·개체 추적이 보이지 않으면 `미확정`으로 남깁니다.

이 파일의 `재감사`는 저장된 모델 출력에 사람이 프레임과 사용자 검토를 대조해 붙인 판정입니다. 현재 파이프라인이 자동으로 아래 숫자를 낸 결과라는 뜻은 아닙니다.


In [ ]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
audit_path = root / "notebooks/smarthome-bench-evidence/state-transition-audit.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
assert len(audit["cases"]) == 7
print("재감사 영상:", ", ".join(c["video_id"] for c in audit["cases"]))


## 결과표

`확정/미확정`은 각 사례의 재감사 판정입니다. `사용자 기준`과 `프레임 잠정 기준`을 구분했습니다. `0936`의 사용자가 센 8동작에는 앞발만 든 2동작도 포함됩니다. 그중 첫 동작은 영상 시작 프레임에 이미 진행 중이므로 엄격한 전후 상태 기준에서는 미확정입니다. `0065`는 사용자에게서 전체 회수만 받았으므로 모델 후보와 실제 사건을 일대일로 대조하지 못했습니다.


In [ ]:
def summarize(case):
    rows = case["candidates"]
    counts = {s: sum(r["status"] == s for r in rows)
              for s in ("confirmed", "uncertain", "duplicate", "rejected")}
    episodes = {(r["subject"], r["episode"]) for r in rows
                if r["status"] == "confirmed"}
    assert len(episodes) == counts["confirmed"]
    for r in rows:
        if r["status"] == "duplicate":
            assert (r["subject"], r["episode"]) in episodes
    assert counts["confirmed"] == case["audit"]["confirmed"]
    assert counts["uncertain"] == case["audit"]["uncertain"]
    assert counts["duplicate"] + counts["rejected"] == case["audit"]["rejected_or_duplicate"]
    assert len(rows) == case["model"]["unique_candidates"]
    return counts

print(f'{"영상":<19} {"모델 원시행":>9} {"재감사 확정":>11} {"미확정":>7} {"중복/오탐":>9}  사람 기준')
for case in audit["cases"]:
    n = summarize(case)
    ref = case["reference"]
    human = f'{ref["confirmed"]}확정+{ref["uncertain"]}미확정'
    if "inclusive_total" in ref:
        human += f' (포함시 {ref["inclusive_total"]})'
    print(f'{case["video_id"]:<19} {case["model"]["raw_rows"]:>9} {n["confirmed"]:>11} {n["uncertain"]:>7} {n["duplicate"]+n["rejected"]:>9}  {human}')


## 후보별 근거

아래 표의 `confirmed`는 수동 대조 후 확인된 전환입니다. `duplicate`는 같은 동물·같은 연속 행동에 해당하고, `rejected`는 목표 상태에 진입하지 않았습니다. `uncertain`은 정답 회수에 억지로 더하지 않습니다.


In [ ]:
for case in audit["cases"]:
    print("\n###", case["video_id"], "—", case["action"])
    print("기준 출처:", case["reference"]["provenance"])
    for row in case["candidates"]:
        print(f'  {row["time"]:>10} | {row["status"]:<9} | {row["subject"]:<16} | {row["evidence"]}')
    if case.get("caveat"):
        print("주의:", case["caveat"])


## 영상 직접 재생

로컬 영상이 있을 때 아래 셀에서 재생할 수 있습니다. `smartbench_0936`은 0초 시작 상태, 13초 오탐, 19초 실제 동작을 확인하세요. `smartbench_0065`는 후보 시각과 각 소파의 실제 착지·이탈을 다시 기록해야 모델과 사용자 회수의 대응을 확정할 수 있습니다.


In [ ]:
from IPython.display import Video, display
video_dir = root / "data/videos"
for case in audit["cases"]:
    if case["video_id"] == "dog_escape":
        continue
    path = video_dir / f'{case["video_id"]}.mp4'
    print(case["video_id"], path if path.exists() else "로컬 영상 없음")
    if path.exists():
        display(Video(filename=str(path), embed=False, html_attributes="controls"))


## 해석과 다음 검증

- `0518`, `0142`: 저장된 모델 후보와 잠정 프레임 검토가 각각 2회로 일치합니다. `0518`은 서로 다른 개체 2마리를 구분해야 합니다.
- `0642`: 모델 원시 3행 중 뒤의 2행은 같은 두 번째 개의 연속 입수입니다. 개체·상태 기준으로 2회입니다. **0.75초 간격으로 무조건 합치는 규칙은 사용하지 않았습니다.**
- `0608`: 11.5~13초 모델 후보는 물 밖 데크에서 걷는 장면이어서 제외했습니다. 재감사 2회입니다.
- `0936`: 모델 상태 후보 9개에서 13초 오탐을 빼고 첫 프레임부터 진행 중인 1개를 미확정으로 두면 **7회 확정 + 1회 미확정**입니다. 사용자 포괄 기준은 8동작입니다. 별도 모델 검증기는 실제 19초를 빼고 13초를 넣어 숫자만 8로 맞췄습니다.
- `0065`: 사용자 기준은 **4회 확실 + 1회 애매**입니다. 모델은 원시 8행(중복 제거 7후보)을 냈지만, 현재 근거로는 그중 2확정·4미확정·1오탐입니다. 이것은 전체 영상 회수 2회라는 뜻이 아닙니다. 사용자 기준 사건별 시각 라벨을 먼저 얻어야 자동 계수 성능을 평가할 수 있습니다.
- `dog_escape`: 사용자 검토 2구간과 모델 원시 5행을 같은 연속 행동별로 묶으면 2회입니다.

잠정 프레임 검토 영상은 사용자가 각 사건의 시각을 독립 확인하기 전까지 확정 정답으로 쓰지 않습니다. 실제 자동화를 개선하려면 모델이 매 후보에 `개체 ID`, `이전 상태`, `이후 상태`, `이탈/리셋 근거`를 구조화해 내고, 누락된 근거는 미확정으로 반환하도록 해야 합니다.
